In [0]:
%run /Workspace/Users/senoom222@gmail.com/databricks-code-repos-senthil/Databricks_workout_2025/Calling_1_wb_to_2_wb_using_util_run/Generic_Specific_Frame

In [0]:
dbutils.widgets.text("Catalog","")
CATALOG = dbutils.widgets.get("Catalog").strip()
dbutils.widgets.text("Schema","")
SCHEMA = dbutils.widgets.get("Schema").strip()

In [0]:
%python
import json

child_output = dbutils.notebook.run("/Workspace/Users/senoom222@gmail.com/databricks-code-repos-senthil/Databricks_workout_2025/Calling_1_wb_to_2_wb_using_util_run/config",120,{"Catalog":CATALOG,"Schema" : SCHEMA})

child_dict = json.loads(child_output)
CATALOG = child_dict["Catalog"]
SCHEMA = child_dict["Schema"]
SRC = child_dict["Source"]
BRONZE = child_dict["Bronze"]
SILVER = child_dict["Silver"]
GOLD = child_dict["Gold"]
SILVERDB = child_dict["Silver Table"]
GOLDDB = child_dict["Gold Table"]


print("Returned Source Location : ",SILVER)
print("Returned Target Location : ",GOLD)
print("Returned GOLDDB Location : ",GOLDDB)

In [0]:
shipments=f"{SILVER}/shipment"
staff=f"{SILVER}/staff"

print(shipments)
print(staff)

SILVERDB = SILVERDB.split('/')[-1]

GOLDDB = GOLDDB.split('/')[-1]

In [0]:
spark.sql(f"""
          CREATE OR REPLACE TEMP VIEW silver_staff_geo_tv
AS
SELECT s.*  
FROM {SILVERDB}.silver_staff s
SEMI JOIN {SILVERDB}.silver_geotag s_geo
ON s.orgin_hub_city = s_geo.city_name
          """)

spark.sql(f"""
          CREATE OR REPLACE TEMP VIEW silver_staff_geo_latlong_tv
AS
SELECT s.*,s_geo.latitude,s_geo.longitude 
FROM silver_staff_geo_tv s
INNER JOIN {SILVERDB}.silver_geotag s_geo
ON s.orgin_hub_city = s_geo.city_name
          """)

In [0]:
spark.sql(f"""
          CREATE OR REPLACE TABLE {GOLDDB}.gold_core_curated_tbl
USING DELTA
AS
SELECT
    s.shipment_id,
    CONCAT(
        SUBSTRING(s.staff_full_name, 1, 2),
        '****',
        SUBSTRING(s.staff_full_name, -1, 1)
    ) AS masked_staff_name,
    s.role,
    s.orgin_hub_city,
    s.latitude,
    s.longitude,
    sh.shipment_cost,
    sh.shipment_year,
    sh.shipment_month,
    sh.route_segment,
    sh.cost_per_kg,
    sh.tax_amount,
    sh.ingestion_timestamp,
    sh.is_expedited,
    sh.is_weekend,
    sh.is_high_value,
    sh.order_prefix,
    sh.order_sequence,
    sh.ship_day,
    sh.route_lane
FROM silver_staff_geo_latlong_tv s
INNER JOIN {SILVERDB}.silver_shipments sh
    USING (shipment_id);
          """)